# 08 · Could a foundation model give a steadier yearly map? TESSERA

**Question:** the module learns change from Esri's yearly land cover, which changes 10–15 % of Guinea a year, a third of it flipping back the next year (notebook 01). Would a map read from TESSERA be steadier, and as accurate?

**TESSERA** (Feng et al. 2026) compresses a full year of Sentinel-1 and Sentinel-2 over each 10 m pixel into 128 numbers, published for the globe from 2017 under CC0. A map read from it uses the whole year at once, where a single-date classification can flicker with the season.

**How it was tested** (`scripts/tessera_lu100.py`, `scripts/tessera_glance_points.py`; outputs `outputs/08_tessera_*.csv`):

* **Blocks.** 24 blocks of 0.1° (about 11 × 11 km), TESSERA's own tiles: 16 forest blocks drawn for the deforestation test and 8 drawn at random across the ecoregions (`data_100m_tools/tessera_blocks.py`). Each 100 m cell gets the mean of its 10 × 10 pixels, 2017–2025.
* **Map.** A linear probe (standardised embeddings, logistic regression, balanced classes), taught Esri's own classes on stable cells: one class in every year 2017–2025 with at least 90 % of the cell in it. Each block is mapped by a probe that never saw it (four block folds).
* **Accuracy** at the 503 GLanCE points of `studies/dataset_checks/02` (independent interpreters, labels 2017–2020), with that notebook's legend and crosswalk. A second probe is taught on the GLanCE points themselves (five folds of 0.5° squares).
* **Stability** with notebook 01's definitions; and **real change**: of the cells that lost at least half their area of forest in a year (Hansen), the share each map moves out of trees between the year before and the year after.

In [1]:
from pathlib import Path
import pandas as pd

OUTS = Path.cwd().parent / "outputs"
acc = pd.read_csv(OUTS / "08_tessera_accuracy.csv")
st = pd.read_csv(OUTS / "08_tessera_stability.csv")
st_g = pd.read_csv(OUTS / "08_tessera_stability_glance_probe.csv")
real = pd.read_csv(OUTS / "08_tessera_real_change.csv")
acc

,Map,Points,Agreement %,95 % CI
0,"Esri, 10 m pixel (notebook 02)",503,75.1,71-79
1,"Esri, 100 m cell (module input)",503,68.0,64-72
2,"TESSERA probe taught by Esri, 10 m pixel",503,75.0,71-79
3,"TESSERA probe taught by Esri, 100 m cell",503,71.8,68-76
4,"TESSERA probe taught by GLanCE (5 folds), 10 m...",503,79.7,76-83
5,"TESSERA probe taught by GLanCE (5 folds), 100 ...",503,76.3,72-80


WorldCover scores 86.3 % at the same points and Dynamic World 77.1 % (`studies/dataset_checks/02`). Esri's class of the 100 m cell, the module's input, agrees less than Esri's 10 m pixel (68.0 % against 75.1 %), because a cell holds a mix of classes.

## 1 · Steadier?

In [2]:
cols = ["changed (% of area)", "of it, back the next year (%)"]
pd.concat([st.groupby("map")[cols].mean(), st_g[st_g["map"] == "TESSERA taught by GLanCE"].groupby("map")[cols].mean()]).round(1)

,changed (% of area),"of it, back the next year (%)"
map,,
Esri (module input),10.1,31.6
TESSERA probe,9.9,35.1
TESSERA taught by GLanCE,20.9,36.5


## 2 · Does it still see real clearing?

In [3]:
real.groupby("map").agg(cells=("cells", "sum"), out_of_trees=("out of trees the year after (%)", "mean")).round(1)

,cells,out_of_trees
map,,
Esri (module input),16394,61.7
TESSERA probe,16411,53.8


## What this means

* **Taught Esri's classes, TESSERA reproduces Esri:** the same accuracy at the GLanCE points (75.0 % against 75.1 % at 10 m), the same change and flips back (9.9 % a year, 35 % back, against 10.1 % and 32 %), and it moves fewer cleared cells out of trees (54 % against 62 %). The stable cells it learns from hold almost no crops (20) or bare ground (3).
* **Taught by independent labels it is more accurate but twice as unsteady:** 76.3 % at 100 m against Esri's 68.0 %, but 20.9 % of the area changes a year. 503 points, most from 2018, are too few for a steady map.
* **So the flicker is the landscape, not the sensor:** the forest–savanna mosaic is hard to class year to year whatever the input, as Dynamic World showed (`studies/dataset_checks/01`). TESSERA is not adopted for the land-use module; Esri stays its input.

### References

* Feng, Z., Atzberger, C., Jaffer, S., Knezevic, J., Sormunen, S., Young, R., Lisaius, M.C., Immitzer, M., Jackson, T., Ball, J., et al. (2026). TESSERA: Temporal embeddings of surface spectra for earth representation and analysis. *CVPR 2026*, 34818–34831. arXiv:2506.20380.
* Karra, K. et al. (2021). Global land use/land cover with Sentinel 2 and deep learning. *IGARSS 2021*, 4704–4707.
* Stanimirova, R. et al. (2023). A global land cover training dataset from 1984 to 2020. *Scientific Data* 10, 879.